# Week 1 · Day 3 — JOINs
**Goal:** Stop working with one table. Connect your data across tables to answer real business questions.

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Day 2 gap to fix today:** Zoom in properly — after confirming a group problem, drill into the breakdown *within* that group using joined tables.

---
## Business First Prompt

> *Your manager asks: "We have high revenue in some regions but I don't know which product categories are driving it — or whether it's a handful of big customers or broad demand. Can you break it down?"*

Write 3–5 sentences below before touching any data:
- What tables would you need to answer this?
- What columns connect those tables?
- What would a useful answer look like — rows, a summary, or both?

**Your answer:**

> I would need the orders table and the order_items table. The column that connects the tables is order_id. A useful answer would look like having a summary including item categories per region and top customers per region.

---
## Setup — run this first

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('week1_ecommerce.db')
print('Connected to week1_ecommerce.db')

Connected to week1_ecommerce.db


---
## Table Preview — run this so you know what you're working with

In [2]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


In [4]:
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)

Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


---
## Concept — Why JOINs Exist

Your data is split across tables for a reason — customers are stored once, not copied into every order row. To answer real questions, you have to reconnect them.

The bridge is a **shared key** — a column that exists in both tables.

```
orders            customers
--------          ----------
order_id          customer_id  ← primary key
customer_id  ────►customer_id  ← foreign key (the bridge)
total_amount      name
region            segment
```

```
orders            order_items
--------          -----------
order_id  ───────►order_id
status            product_name
region            category
                  unit_price
                  quantity
```

**Rule:** Before writing a JOIN, find the shared column. No shared column = no valid join.

---
## Concept — INNER JOIN

Returns only rows that have a match in **both** tables. If a customer has no orders, they disappear. If an order has no matching customer, it disappears.

```sql
SELECT o.order_id,
       o.total_amount,
       c.name,
       c.segment
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
```

**Aliases:** `orders o` means you write `o.column` instead of `orders.column`. Always alias when joining — keeps queries readable and prevents ambiguity errors.

In [5]:
# EXAMPLE — join orders to customers, see name + segment alongside order data
q = """
SELECT o.order_id,
       o.order_date,
       o.total_amount,
       o.status,
       c.name,
       c.segment
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
LIMIT 10
"""

df = pd.read_sql_query(q, conn)
display(df)

,order_id,order_date,total_amount,status,name,segment
0,1001,2024-01-05,250.0,completed,Alice Martin,VIP
1,1003,2024-01-20,430.5,completed,Sara Lopez,Regular
2,1005,2024-02-14,310.0,completed,Alice Martin,VIP
3,1007,2024-03-15,620.0,completed,Bob Chen,Regular
4,1008,2024-04-02,190.0,completed,Sara Lopez,Regular
5,1010,2024-05-05,510.0,completed,James Wu,VIP
6,1012,2024-06-08,275.0,completed,Priya Nair,New
7,1013,2024-06-25,480.0,completed,Omar Hassan,Regular
8,1015,2024-07-28,390.0,completed,Alice Martin,VIP
9,1016,2024-08-05,145.0,completed,David Kim,New


---
## Drill 1 — Order details with customer name

Show the 20 most recent completed orders with the customer's name and segment alongside the order data.  
Columns: order_id, order_date, total_amount, name, segment.

Tables: `orders` + `customers` · Join key: `customer_id`  
Clauses: `SELECT` · `FROM` · `INNER JOIN` · `WHERE` · `ORDER BY` · `LIMIT`

In [104]:

q1 = """ SELECT o.order_id,
       o.order_date,
       o.total_amount,
       c.name,
       c.segment
    FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    ORDER BY o.order_date DESC
    LIMIT 20

"""

df = pd.read_sql_query(q1, conn)
display(df)



,order_id,order_date,total_amount,name,segment
0,1030,2024-12-28,640.0,Sara Lopez,Regular
1,1029,2024-12-18,290.0,Bob Chen,Regular
2,1027,2024-12-02,980.0,Carlos Mendez,VIP
3,1026,2024-11-25,415.0,Emma Rossi,Regular
4,1024,2024-11-03,335.0,Lily Thompson,New
5,1023,2024-10-22,720.0,Omar Hassan,Regular
6,1021,2024-10-07,560.0,James Wu,VIP
7,1019,2024-09-14,165.0,Bob Chen,Regular
8,1018,2024-09-02,890.0,Carlos Mendez,VIP
9,1016,2024-08-05,145.0,David Kim,New


'Missing' is not recognized as an internal or external command,
operable program or batch file.


---
## Drill 2 — Order line items with order status

Show each order item alongside the order's status and region.  
Columns: item_id, product_name, category, quantity, unit_price, status, region. Limit 20.

Tables: `order_items` + `orders` · Join key: `order_id`  
Clauses: `SELECT` · `FROM` · `INNER JOIN` · `LIMIT`

In [105]:
q2 = """SELECT oi.item_id,
        o.order_id,
        oi.category,
        oi.product_name,
        oi.quantity,
        oi.unit_price,
        o.status,
        o.region
FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
LIMIT 20

"""

df = pd.read_sql_query(q2, conn)
display(df)

,item_id,order_id,category,product_name,quantity,unit_price,status,region
0,1,1001,Electronics,Wireless Mouse,1,45.00,completed,West
1,2,1001,Electronics,USB Hub,2,25.00,completed,West
2,3,1001,Accessories,Desk Mat,1,35.00,completed,West
3,4,1002,Accessories,Phone Stand,1,22.99,cancelled,East
4,5,1003,Electronics,Mechanical Keyboard,1,120.00,completed,West
5,6,1003,Electronics,Monitor Light,1,55.00,completed,West
6,7,1003,Accessories,Cable Organizer,3,18.50,completed,West
7,8,1004,Electronics,Webcam,1,89.00,refunded,North
8,9,1004,Electronics,Ring Light,1,65.00,refunded,North
9,10,1005,Furniture,Standing Desk,1,310.00,completed,East


---
## Drill 3 — High-value cancelled orders with customer name

Show cancelled orders over $400 with the customer's name and segment.  
Columns: order_id, order_date, total_amount, region, name, segment. Highest amount first.

Tables: `orders` + `customers` · Join key: `customer_id`

In [19]:
q3 = """SELECT o.order_id,
        c.name,
        c.segment,
        o.order_date,
        o.total_amount,
        o.status,
        o.region
        FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.total_amount > 400 AND o.status = 'cancelled'
        ORDER BY o.total_amount DESC
    
"""

df = pd.read_sql_query(q3, conn)
display(df)

,order_id,name,segment,order_date,total_amount,status,region


---
## Concept — LEFT JOIN

Returns **all rows from the left table**, and matches from the right where they exist. No match = NULLs on the right side.

Use LEFT JOIN when you want to keep everything from one table — even records with no match.

```sql
SELECT c.customer_id,
       c.name,
       o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
-- customers with no orders still appear, with NULL for order_id
```

**When to use each:**
| Goal | Use |
|------|-----|
| Only matched records | INNER JOIN |
| All left records, matched or not | LEFT JOIN |
| Find records with NO match | LEFT JOIN + WHERE right.id IS NULL |

In [8]:
# EXAMPLE — find customers who have never placed an order
q = """
SELECT c.customer_id,
       c.name,
       c.segment,
       o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
LIMIT 10
"""

df = pd.read_sql_query(q, conn)
display(df)

,customer_id,name,segment,order_id


---
## Drill 4 — Customers with no orders

Find all customers who have never placed any order.  
Columns: customer_id, name, segment, country.

Tables: `customers` + `orders` · Join key: `customer_id`  
Hint: after a LEFT JOIN, unmatched rows have NULL in the right table's columns.

In [113]:

q4 = """ SELECT c.customer_id,
        c.name,
        c.segment,
        c.country
        FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_id IS NULL

"""

df = pd.read_sql_query(q4, conn)
display(df)

,customer_id,name,segment,country


---
## Drill 5 — All customers with their order count (include zero-order customers)

Show every customer and how many orders they've placed — including customers with 0 orders.  
Columns: name, segment, country, order_count. Sort by order_count descending.

Hint: `COUNT(o.order_id)` — counting the right table's column returns 0 for NULLs, not 1. `COUNT(*)` would count the NULL row as 1.

In [19]:

q5 = """ SELECT c.customer_id,
        c.name,
        c.segment,
        c.country,
        COUNT(o.order_id) AS order_count
        FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
         GROUP BY c.customer_id, c.name, c.segment, c.country
        ORDER BY order_count DESC

        """

df = pd.read_sql_query(q5, conn)
display(df)

,customer_id,name,segment,country,order_count
0,1,Alice Martin,VIP,US,4
1,2,Bob Chen,Regular,CA,4
2,3,Sara Lopez,Regular,US,4
3,4,James Wu,VIP,UK,3
4,5,Priya Nair,New,US,3
5,6,Omar Hassan,Regular,CA,3
6,7,Lily Thompson,New,US,3
7,8,David Kim,New,AU,2
8,9,Emma Rossi,Regular,US,2
9,10,Carlos Mendez,VIP,MX,2


---
## Concept — JOIN + GROUP BY Together

JOIN to bring in context from another table, then GROUP BY to summarize across that context.

```sql
SELECT c.segment,
       COUNT(*) as total_orders,
       SUM(o.total_amount) as total_revenue
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment
ORDER BY total_revenue DESC
```

**Full clause order with JOIN:**
```sql
SELECT
FROM
JOIN ... ON
WHERE
GROUP BY
HAVING
ORDER BY
LIMIT
```

In [21]:
# EXAMPLE — revenue by customer segment
q = """
SELECT c.segment,
       COUNT(*) as total_orders,
       SUM(o.total_amount) as total_revenue,
       ROUND(AVG(o.total_amount), 2) as avg_order_value
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment
ORDER BY total_revenue DESC
"""

df = pd.read_sql_query(q, conn)
display(df)

,segment,total_orders,total_revenue,avg_order_value
0,Regular,9,3950.5,438.94
1,VIP,7,3890.0,555.71
2,New,3,755.0,251.67


---
## Drill 6 — Revenue by product category

Show total revenue and total units sold per product category — completed orders only.  
Columns: category, total_revenue, total_units. Highest revenue first.

Tables: `orders` + `order_items` · Join key: `order_id`

In [119]:
q6 = """ SELECT oi.category,
        SUM(o.total_amount) AS total_revenue,
        SUM(oi.quantity) AS total_units
        FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed' 
        Group BY oi.category
        ORDER BY total_revenue DESC
        
"""

df = pd.read_sql_query(q6, conn)
display(df)

,category,total_revenue,total_units
0,Electronics,8151.0,19
1,Accessories,4890.5,15
2,Furniture,2750.0,5


---
## Drill 7 — Top 10 customers by total spend

Show each customer's name, segment, total orders placed, and total spend — completed only. Top 10 by spend.

Tables: `orders` + `customers` · Join key: `customer_id`  
Note: GROUP BY `c.customer_id, c.name, c.segment` — include all non-aggregate SELECT columns.

In [40]:

q7 = """ SELECT c.name,
        c.segment,
        SUM(o.total_amount) AS total_spend,
        COUNT(*) AS total_orders_placed
        FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id, c.name, c.segment
        ORDER BY total_spend DESC
        LIMIT 10



"""

df = pd.read_sql_query(q7, conn)
display(df)

,name,segment,total_spend,total_orders_placed
0,Carlos Mendez,VIP,1870.0,2
1,Sara Lopez,Regular,1260.5,3
2,Omar Hassan,Regular,1200.0,2
3,Bob Chen,Regular,1075.0,3
4,James Wu,VIP,1070.0,2
5,Alice Martin,VIP,950.0,3
6,Emma Rossi,Regular,415.0,1
7,Lily Thompson,New,335.0,1
8,Priya Nair,New,275.0,1
9,David Kim,New,145.0,1


---
## Drill 8 — Average order value by customer segment and region

Show average order value broken out by both segment AND region — completed orders only.  
Columns: segment, region, avg_order_value, total_orders. Sort by segment, then avg_order_value descending.

Tables: `orders` + `customers` · Join key: `customer_id`

In [128]:
q8 = """SELECT   c.segment,
        o.region,
        ROUND (AVG(o.total_amount),1) AS avg_order_value,
        COUNT(*) AS total_orders
        FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.segment, o.region
        ORDER BY c.segment, avg_order_value DESC

"""

df = pd.read_sql_query(q8, conn)
display(df)

,segment,region,avg_order_value,total_orders
0,New,North,335.0,1
1,New,South,210.0,2
2,Regular,South,517.5,2
3,Regular,West,492.1,5
4,Regular,North,227.5,2
5,VIP,West,615.0,2
6,VIP,South,560.0,1
7,VIP,East,530.0,3
8,VIP,North,510.0,1


---
## Drill 9 — Top 10 products by units sold

Show the top 10 most ordered product names by total units sold — completed orders only.  
Columns: product_name, total_units_sold, total_revenue.

Tables: `order_items` + `orders` · Join key: `order_id`  
Hint: use `SUM(i.quantity * i.unit_price)` for item-level revenue.

In [122]:
q9 = """ SELECT oi.product_name,
        SUM(oi.quantity) AS total_quantity_sold, 
        SUM(oi.quantity * oi.unit_price) AS total_revenue
        FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY oi.product_name
        ORDER BY total_quantity_sold DESC
        LIMIT 10

"""

df = pd.read_sql_query(q9, conn)
display(df)

,product_name,total_quantity_sold,total_revenue
0,Cable Organizer,3,55.5
1,Wrist Rest,2,56.0
2,Wireless Charger,2,70.0
3,USB Hub,2,50.0
4,Standing Desk,2,620.0
5,Mechanical Keyboard,2,240.0
6,HDMI Cable,2,30.0
7,Wireless Mouse,1,45.0
8,Wireless Keyboard,1,95.0
9,Ultrawide Monitor,1,650.0


---
## Concept — Chaining Multiple JOINs

You can join more than two tables by chaining JOIN clauses. Each JOIN adds one more table to the query.

```sql
SELECT c.segment,
       i.category,
       SUM(i.quantity * i.unit_price) as total_revenue
FROM orders o
INNER JOIN order_items i ON o.order_id = i.order_id
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment, i.category
```

**Rule:** Each JOIN needs its own ON condition. Think of it as adding one table at a time — `orders` connects to `order_items` via order_id, then `orders` also connects to `customers` via customer_id.

In [54]:
# EXAMPLE — revenue by customer segment using all three tables
q = """
SELECT c.segment,
       i.category,
       ROUND(SUM(i.quantity * i.unit_price), 2) as total_revenue,
       SUM(i.quantity) as total_units
FROM orders o
INNER JOIN order_items i ON o.order_id = i.order_id
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment, i.category
ORDER BY c.segment, total_revenue DESC
"""

df = pd.read_sql_query(q, conn)
display(df)

,segment,category,total_revenue,total_units
0,New,Electronics,489.0,3
1,Regular,Electronics,1055.0,9
2,Regular,Furniture,785.0,3
3,Regular,Accessories,271.5,9
4,VIP,Electronics,1540.0,7
5,VIP,Furniture,730.0,2
6,VIP,Accessories,246.0,6


---
## Drill 10 — Category breakdown by region (three tables)

Show revenue per region AND per category — completed orders only.  
Columns: region, category, total_revenue. Sort by region, then total_revenue descending.

Tables: `orders` + `order_items` · Join key: `order_id`  
Hint: only two tables needed here — `customers` is not required.

In [125]:
q10 = """ SELECT oi.category, 
        o.region,
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        COUNT(*) AS total_orders
        FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY oi.category, o.region
        ORDER BY o.region, total_revenue DESC
"""

df = pd.read_sql_query(q10, conn)
display(df)

,category,region,total_revenue,total_orders
0,Electronics,East,770.0,2
1,Furniture,East,310.0,1
2,Accessories,East,110.0,2
3,Electronics,North,360.0,3
4,Accessories,North,91.0,2
5,Furniture,South,895.0,3
6,Electronics,South,364.0,3
7,Accessories,South,65.0,1
8,Electronics,West,1590.0,9
9,Furniture,West,310.0,1


---
## Drill 11 — Segments with high cancellation value (JOIN + HAVING)

Find customer segments where total cancelled order value exceeds $500.  
Columns: segment, cancelled_orders, total_cancelled_value.

Tables: `orders` + `customers` · Join key: `customer_id`  
Clauses: `SELECT` · `FROM` · `INNER JOIN` · `WHERE` · `GROUP BY` · `HAVING` · `ORDER BY`

In [69]:
q11 = """ SELECT c.segment,
        COUNT(*) AS cancelled_orders,
        SUM(o.total_amount) AS total_cancelled_value
        FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status = 'cancelled'
        GROUP BY c.segment
        HAVING SUM(o.total_amount) > 500
        ORDER BY total_cancelled_value DESC

"""

df = pd.read_sql_query(q11, conn)
display(df)

,segment,cancelled_orders,total_cancelled_value


---
## Block 2 · 11:00am — Applied Challenge

> *Same manager, continuing from Day 2: "OK so West is leading in revenue — but I need to know WHY. Is it a specific product category? A particular customer segment? Or are all categories performing well there?"*

This is a root cause question. You need joins to answer it.

**Query 1:** What product categories drive West's revenue — and how does that compare to other regions?  
**Query 2:** What customer segments are buying in West — and are they the same segments spending in other regions?

Write your framing sentence first.

**My approach — what tables do I need and why:**

> My approach to find the root cause of West region's lead in revenue will be by identifying leading categories per customer segment. This will help management identify what is succeeding and apply that insight to the other regions that are underperforming

In [ ]:
# Query 1 — West revenue by product category vs other regions
q12 = """ SELECT o.region,
                oi.category,
         SUM(oi.quantity * oi.unit_price) AS total_revenue,
        COUNT(*) AS total_orders
        FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY oi.category, o.region
        ORDER BY CASE WHEN  o.region = 'West' THEN 0 ELSE 1 END , o.region, total_revenue DESC

"""

df = pd.read_sql_query(q12, conn)
display(df)

,region,category,total_revenue,total_orders
0,West,Electronics,1590.0,9
1,West,Furniture,310.0,1
2,West,Accessories,251.5,6
3,East,Electronics,770.0,2
4,East,Furniture,310.0,1
5,East,Accessories,110.0,2
6,North,Electronics,360.0,3
7,North,Accessories,91.0,2
8,South,Furniture,895.0,3
9,South,Electronics,364.0,3


In [82]:
# Query 2 — customer segment breakdown by region
q13 = """ SELECT o.region,
        c.segment,
        COUNT(*) AS total_orders,
        SUM(o.total_amount) AS total_revenue
        FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id   
        WHERE o.status = 'completed'
        GROUP BY o.region, c.segment
        ORDER BY CASE WHEN o.region = 'West' THEN 0 ELSE 1 END, o.region, total_revenue DESC

"""

df = pd.read_sql_query(q13, conn)
display(df)

,region,segment,total_orders,total_revenue
0,West,Regular,5,2460.5
1,West,VIP,2,1230.0
2,East,VIP,3,1590.0
3,North,VIP,1,510.0
4,North,Regular,2,455.0
5,North,New,1,335.0
6,South,Regular,2,1035.0
7,South,VIP,1,560.0
8,South,New,2,420.0


In [5]:
#Own query- AVG cost per order

q14 = """ SELECT o.region,
            oi.category,
            SUM(oi.quantity * oi.unit_price) AS total_revenue,
        ROUND(AVG(oi.quantity * oi.unit_price),2) AS avg_order_value,
        COUNT(*) AS total_orders,
        c.segment

            FROM orders o INNER JOIN order_items oi ON o.order_id = oi.order_id
            INNER JOIN customers c ON o.customer_id = c.customer_id
            WHERE o.status = 'completed' AND order_date >= '2024-09-30' AND order_date < '2024-12-31'
            GROUP BY o.region, oi.category, c.segment
            ORDER BY CASE WHEN o.region = 'West' THEN 0 ELSE 1 END, o.region, avg_order_value DESC
"""


df = pd.read_sql_query(q14, conn)
display(df)

,region,category,total_revenue,avg_order_value,total_orders,segment
0,West,Electronics,580.0,580.00,1,VIP
1,West,Furniture,310.0,310.00,1,Regular
2,West,Electronics,670.0,223.33,3,Regular
3,West,Accessories,45.0,45.00,1,VIP
4,North,Electronics,220.0,220.00,1,New
5,North,Electronics,45.0,45.00,1,Regular
6,South,Furniture,420.0,420.00,1,VIP
7,South,Electronics,95.0,95.00,1,Regular
8,South,Accessories,65.0,65.00,1,Regular


**What does the data say now?**  
What's driving West's performance — categories, segments, or both? What would you tell your manager?

> West is outperforming all other regions. The Regular segment is the primary driver with $2,460 in revenue across 5 orders. Electronics is the top category at $1,590, more than double East's $770. At the item level, VIP Electronics orders average $580, the highest single avg order value in the dataset. Both segment mix and category concentration explain West's lead.

---
## Capstone Drill — Full root cause analysis in one query

Your manager wants a single table showing: for each region and customer segment, what is the total revenue, total orders, and average order value — completed orders only. Sort by region, then total_revenue descending.

This requires: `INNER JOIN` (two tables) + `WHERE` + `GROUP BY` (two columns) + `ROUND` + `ORDER BY` (two columns).

Tables: `orders` + `customers`

In [124]:
q15 = """ SELECT o.region,
             c.segment,
                        SUM(o.total_amount) AS total_revenue,
        ROUND(AVG(o.total_amount),2) AS avg_order_value,
        COUNT(*) AS total_orders
       
            FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id
            WHERE o.status = 'completed' AND order_date >= '2024-09-30' AND order_date < '2024-12-31'
            GROUP BY o.region, c.segment
            ORDER BY CASE WHEN o.region = 'West' THEN 0 ELSE 1 END, o.region, total_revenue DESC
"""


df = pd.read_sql_query(q15, conn)
display(df)

,region,segment,total_revenue,avg_order_value,total_orders
0,West,Regular,1360.0,680.0,2
1,West,VIP,980.0,980.0,1
2,North,New,335.0,335.0,1
3,North,Regular,290.0,290.0,1
4,South,VIP,560.0,560.0,1
5,South,Regular,415.0,415.0,1


---
## Day 3 Checkpoint — answer before Day 4

Plain English — no SQL needed.

**1. What is the difference between INNER JOIN and LEFT JOIN? Give a real business example of when you'd use each.**

> INNER JOIN returns only rows where a match exists in both tables. LEFT JOIN returns all rows from the left table, with NULLs where no match exists in the right table. I would use INNER JOIN when I need necessary data like completed status and order category. I would use LEFT JOIN when im looking for customers that have never placed an order but are registered.

**2. Your manager asks: "Show me all customers and how much they've spent — including customers who haven't bought anything yet." Which join type do you use and why?**

> I would use LEFT JOIN because if they have yet to buy anything that value would be NULL.

**3. You're joining orders to order_items. Before writing a single line of SQL — what column connects the two tables, and what does a row look like after the join?**

> The column that connects both tables is order_id, the column in common. After the join, each row represents one line item, with order-level columns (region, status, date) repeated for every item in that order. So if an order has 3 items, it appears as 3 rows.

**4. The capstone drill used two columns in GROUP BY. Why — what breaks if you only GROUP BY one of them?**

> If you GROUP BY only region when you also have segment in your SELECT, SQL doesn't know how to collapse multiple segments into one row, it either errors or returns unpredictable results. You need to GROUP BY every non-aggregated column in SELECT.